In [2]:
from PIL import Image
import numpy as np
from scipy.ndimage import uniform_filter

preds_path = "../v3+_shadowed_512_128_full_prediction_map.npy"
preds_raw = np.load(preds_path)

smoothed = uniform_filter(preds_raw, size=5)

# binarize
preds_bin = (smoothed > 0.6149).astype(np.uint8)

# save to uint8
img = (preds_bin * 255).astype(np.uint8)
img_raw = (preds_raw * 255).astype(np.uint8)

Image.fromarray(img).save("../predictions_map.png")
Image.fromarray(img_raw).save("../probabilities_map.png")

In [5]:
import rasterio
import numpy as np
import copy

origPath = '/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/QGIS/250227_Bundzel_Sperka_Terasy/Hradisko_Ladzany_DEM25cm.tif'

pngPath = '/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/v3+_shadowed_512_128_probabilities_map.png'

trgtPath = pngPath[:-4] + '.tif'

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
#data = np.array(Image.open(pngPath))
data = np.array(Image.open(pngPath)).astype(np.float32) / 255.0

#data = (data>0).astype(np.uint8)

print(data.dtype)
print(data.shape)
print(np.min(data))
print(np.max(data))

# selection ordering (column min, row min, column max, row max)
selection = [0, 0, data.shape[1], data.shape[0]]

orig = rasterio.open(origPath)

bounds = orig.bounds,
transform = orig.transform,
crs = orig.crs

origTransform = copy.deepcopy(transform)



[left,top] = orig.transform * (selection[0],selection[1])

[right,bottom] = orig.transform * (selection[2],selection[3])


print(origTransform)
newTransform = rasterio.transform.Affine.translation(left, top)\
    * rasterio.transform.Affine.scale(origTransform[0][0], origTransform[0][4])
print(newTransform)

trgt = rasterio.open(
        trgtPath,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=crs,
        transform=newTransform,
        nodata = 255
        )


trgt.write(data, 1)


trgt.close()
orig.close()




float32
(5411, 2895)
0.0
0.9764706
(Affine(0.25, 0.0, -441915.25,
       0.0, -0.25, -1274107.5),)
| 0.25, 0.00,-441915.25|
| 0.00,-0.25,-1274107.50|
| 0.00, 0.00, 1.00|
